# Create Sample Data for Ecommerce Streaming Demo

In [0]:
%sql
SHOW EXTERNAL LOCATIONS

name,url,comment
adb_ai,abfss://unity-catalog-storage@dbstoragedyeqybyx3jqyi.dfs.core.windows.net/1658366818772244,null
adb_dev,abfss://unity-catalog-storage@dbstoragecvkffyaggkqri.dfs.core.windows.net/279469438045347,null
adb_train,abfss://unity-catalog-storage@dbstorage3tzw5vtlo3p7m.dfs.core.windows.net/1283281679973198,null
bronze-sa,abfss://bronze@dbstoragedyeqybyx3jqyi.dfs.core.windows.net/,null
databricks_train,abfss://unity-catalog-storage@dbstoragebtslppzvwity6.dfs.core.windows.net/3214080724152465,null
healthsaadlsrheus_outbound,abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/,Access to healthdata container on healthsaadlsrheus
test,abfss://unity-catalog-storage@dbstorageswideqgakkcy6.dfs.core.windows.net/2953091036516858,null


In [0]:
# ADLS Gen2 configuration - use healthdata container in healthsaadlsrheus storage account
STORAGE_ACCOUNT = 'healthsaadlsrheus'
CONTAINER       = 'healthdata'
SOURCE_PATH     = 'ecommerce/transactions'

# Full ADLS Gen2 source path (Auto Loader reads from here)
ADLS_SOURCE = f'abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{SOURCE_PATH}'

# Checkpoint location (decoupled from workspace — stored in ADLS)
CHECKPOINT_PATH = 'checkpoints/bronze_ecommerce_transactions'
CHECKPOINT_LOCATION = f'abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{CHECKPOINT_PATH}'

# Target catalog and table
CATALOG = 'bronze'
SCHEMA  = 'ecommerce'
TABLE   = 'transactions'

print(f'Source:     {ADLS_SOURCE}')
print(f'Checkpoint: {CHECKPOINT_LOCATION}')
print(f'Target:     {CATALOG}.{SCHEMA}.{TABLE}')

Source:     abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/ecommerce/transactions
Checkpoint: abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions
Target:     bronze.ecommerce.transactions


In [0]:
dbutils.secrets.listScopes()

[SecretScope(name='adf-pipeline'),
 SecretScope(name='akv-test'),
 SecretScope(name='databricks-kv-rh-scope'),
 SecretScope(name='kvfabricprodeus2rh')]

In [0]:
# Verify access via the external location
try:
    files = dbutils.fs.ls(ADLS_SOURCE)
    print(f'Access verified via external location')
    print(f'{ADLS_SOURCE}')
except Exception as e:
    print(f'Cannot access {ADLS_SOURCE}')
    print(f'Error: {e}')
    print('External location must be properly setup before continuing')

Access verified via external location
abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/ecommerce/transactions


In [0]:
existing_catalogs = [row[0] for row in spark.sql('SHOW CATALOGS').collect()]

# check if expected catalog exists and if not create it
if CATALOG not in existing_catalogs:
    managed_loc = f'abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{CATALOG}'
    spark.sql(f'CREATE CATALOG {CATALOG} MANAGED LOCATION "{managed_loc}"')

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')
spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')

print(f'Using {CATALOG}.{SCHEMA}')

Using bronze.ecommerce


In [0]:
import random
import uuid
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql.types import (
  StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

"""
PRODUCTS = [
  ('PROD-001', 'Wireless Mouse', 29.99, 'Electronics'),
  ('PROD-002', 'USB-C Hub', 49.99, 'Electronics'),
  ('PROD-003', 'Mechanical Keyboard', 89.99, 'Electronics'),
  ('PROD-004', 'Monitor Stand', 39.99, 'Office'),
  ('PROD-005', 'Desk Lamp', 24.99, 'Office'),
  ('PROD-006', 'Notebook (A5)', 12.99, 'Stationery'),
  ('PROD-007', 'Webcam HD', 59.99, 'Electronics'),
  ('PROD-008', 'Headphones', 79.99, 'Electronics'),
  ('PROD-009', 'Mouse Pad XL', 19.99, 'Accessories'),
  ('PROD-010', 'Laptop Backpack', 54.99, 'Accessories'),
]
"""

PRODUCTS = [
  ('PROD-011', 'Bluetooth Keyboard', 64.99, 'Electronics'),
  ('PROD-012', 'Portable SSD 1TB', 129.99, 'Electronics'),
  ('PROD-013', 'Ergonomic Mouse', 49.99, 'Electronics'),
  ('PROD-014', 'Standing Desk Converter', 189.99, 'Office'),
  ('PROD-015', 'Whiteboard (Magnetic)', 79.99, 'Office'),
  ('PROD-016', 'Planner Notebook (A4)', 18.99, 'Stationery'),
  ('PROD-017', 'USB Desk Microphone', 99.99, 'Electronics'),
  ('PROD-018', 'Noise Cancelling Earbuds', 149.99, 'Electronics'),
  ('PROD-019', 'Laptop Sleeve 15-inch', 34.99, 'Accessories'),
  ('PROD-020', 'Cable Organizer Kit', 22.99, 'Accessories'),
]



CUSTOMERS = [f'CUST-{i:04d}' for i in range(1, 51)]
REGIONS = ['East', 'West', 'Central', 'South', 'North']

NUM_RECORDS = 200
batch_id = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
base_time = datetime.utcnow() - timedelta(hours=1)

rows = []
for i in range(NUM_RECORDS):
  product = random.choice(PRODUCTS)
  quantity = random.randint(1, 5)
  rows.append(Row(
    order_id=str(uuid.uuid4()),
    product_id=product[0],
    product_name=product[1],
    category=product[3],
    unit_price=product[2],
    quantity=quantity,
    total_amount=round(product[2] * quantity, 2),
    customer_id=random.choice(CUSTOMERS),
    region=random.choice(REGIONS),
    order_timestamp=base_time + timedelta(seconds=random.randint(0, 3600)),
  ))

schema = StructType([
  StructField('order_id', StringType(), False),
  StructField('product_id', StringType(), False),
  StructField('product_name', StringType(), False),
  StructField('category', StringType(), False),
  StructField('unit_price', DoubleType(), False),
  StructField('quantity', IntegerType(), False),
  StructField('total_amount', DoubleType(), False),
  StructField('customer_id', StringType(), False),
  StructField('region', StringType(), False),
  StructField('order_timestamp', TimestampType(), False),
])

df_sample = spark.createDataFrame(rows, schema)

output_path = f'{ADLS_SOURCE}/batch_{batch_id}'
df_sample.write.mode('overwrite').option('header', 'true').csv(output_path)

print(f'Wrote {NUM_RECORDS} records to {output_path}')
df_sample.show(5, truncate=False)

/home/spark-8c754674-963a-410a-92fa-b4/.ipykernel/2691/command-7075199843521586-3283869515:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  batch_id = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
/home/spark-8c754674-963a-410a-92fa-b4/.ipykernel/2691/command-7075199843521586-3283869515:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time = datetime.utcnow() - timedelta(hours=1)


Wrote 200 records to abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/ecommerce/transactions/batch_20260309_145226
+------------------------------------+----------+-----------------------+-----------+----------+--------+------------+-----------+-------+--------------------------+
|order_id                            |product_id|product_name           |category   |unit_price|quantity|total_amount|customer_id|region |order_timestamp           |
+------------------------------------+----------+-----------------------+-----------+----------+--------+------------+-----------+-------+--------------------------+
|7f2f59dc-031d-448c-a283-2243f6dfc3d1|PROD-014  |Standing Desk Converter|Office     |189.99    |3       |569.97      |CUST-0019  |Central|2026-03-09 13:58:56.897848|
|2e5852a0-dbfa-45a9-a48a-d522b8f0f35b|PROD-012  |Portable SSD 1TB       |Electronics|129.99    |5       |649.95      |CUST-0022  |North  |2026-03-09 13:57:15.897848|
|34f07a04-0890-4893-b18e-2d94ab6cb0ba|PROD-020